# Five sales measures: eigenvalues of the correlations

**Eigenvalue Decomposition** · a worked example from the StatNex teaching guide

**The story.** A sales manager has the correlation matrix of five measures for 20 sales reps: meetings, emails, deals closed, revenue and customer rating. Here it is split into its eigenvalues and eigenvectors, the engine behind principal component analysis.

**The question.** How many independent "directions" of performance are really in these five measures?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `12_utility/eigen_decomposition.csv` (5 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Measure,Meetings per week,Emails per day,Deals closed,Revenue ($000),Customer rating
Meetings per week,1.0,0.3586,0.7429,0.7164,0.1357
Emails per day,0.3586,1.0,0.1832,0.1392,-0.2765
Deals closed,0.7429,0.1832,1.0,0.9462,0.149
Revenue ($000),0.7164,0.1392,0.9462,1.0,0.1354
Customer rating,0.1357,-0.2765,0.149,0.1354,1.0

## Libraries

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## The matrix

In [ ]:
# A symmetric matrix, one column per variable; a blank cell counts as 0. When only one
# triangle was typed (the other left blank), it is mirrored to make the matrix
# symmetric, as the app does.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Meetings per week", "Emails per day", "Deals closed", "Revenue ($000)", "Customer rating"]
A = data[columns].fillna(0).to_numpy(float)
upper = A[np.triu_indices(len(A), 1)]
lower = A[np.tril_indices(len(A), -1)]
if not upper.any() or not lower.any():
    A = A + A.T - np.diag(np.diag(A))

## Eigenvalues

In [ ]:
# Each eigenvalue is the variance along one direction (its eigenvector). The
# eigenvalues add up to the trace (for a correlation matrix, the number of variables)
# and multiply to the determinant. A large one marks a direction most of the
# variation lies along, which is what principal component analysis keeps; one near 0
# marks a combination of the variables that hardly varies at all, meaning some of
# them nearly repeat the others (the overlap a large VIF also flags).
# eigh is the routine for symmetric matrices: real eigenvalues, orthogonal vectors.
values, vectors = np.linalg.eigh(A)
# Largest first.
order = np.argsort(values)[::-1]
values, vectors = values[order], vectors[:, order]
axes = [str(i + 1) for i in range(len(values))]
print(pd.DataFrame([values], index=["eigenvalue"], columns=axes))

## Eigenvectors

In [ ]:
# An eigenvector's sign is arbitrary (v and -v are both eigenvectors), so each is
# turned to make its elements sum to a positive number -- the app's rule.
for j in range(vectors.shape[1]):
    if vectors[:, j].sum() < 0:
        vectors[:, j] = -vectors[:, j]
print()
print(pd.DataFrame(vectors, index=columns, columns=axes))

## Contribution

In [ ]:
# Each eigenvalue's share of the total -- meaningful only when none is negative.
if values.min() >= -1e-12 and values.sum() > 0:
    share = values / values.sum()
    contribution = pd.DataFrame([share, np.cumsum(share)],
                                index=["contribution", "cumulative"], columns=axes)
    print()
    print(contribution)
else:
    print("\nSome eigenvalues are negative: the matrix is not positive definite.")

## What the result says

An eigenvalue says how much of the differences between reps one independent direction carries. An eigenvector says which measures make up that direction. The five eigenvalues are 2.72, 1.29, 0.67, 0.27 and 0.05.

The contribution rate is each eigenvalue's share of the total. The first direction carries 54%. Its eigenvector is large for meetings (0.54), deals (0.58) and revenue (0.57), so it is a general "selling" direction. The first two directions together carry 80%, and the first four carry 99%.

The fifth carries only 1%. Its eigenvector sets deals (-0.72) against revenue (0.69): these two measures almost always move together, so their difference holds almost nothing. The five measures hold about four independent directions of performance, not five.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.